# Urban vs Non-Urban Dummy Variable Creation


In [38]:
import pandas as pd
from scipy.spatial import cKDTree

In [39]:
cur_data = pd.read_csv("../../data/Fall 2024 data/cleaned_alldata_version2.csv")
pop_data = pd.read_csv("../../data/spring_2025_data/india-population-data.csv")
print("Fa24 clean data columns: ", cur_data.columns)
print("India population data columns: ", pop_data.columns)

Fa24 clean data columns:  Index(['Unnamed: 0', 'Date_of_observation', 'Species_name', 'Lat', 'Long',
       'State_name', 'Leaves_fresh', 'Leaves_mature', 'Leaves_old',
       'Flowers_bud', 'Flowers_open', 'Flowers_male', 'Flowers_Female',
       'Fruits_unripe', 'Fruits_ripe', 'Fruits_open', 'Year', 'Week',
       'Species_id'],
      dtype='object')
India population data columns:  Index(['GUBID', 'ISOALPHA', 'COUNTRYNM', 'NAME1', 'NAME2', 'NAME3', 'NAME4',
       'NAME5', 'NAME6', 'CENTROID_X', 'CENTROID_Y', 'INSIDE_X', 'INSIDE_Y',
       'CONTEXT', 'CONTEXT_NM', 'WATER_CODE', 'TOTAL_A_KM', 'WATER_A_KM',
       'LAND_A_KM', 'UN_2000_E', 'UN_2005_E', 'UN_2010_E', 'UN_2015_E',
       'UN_2020_E', 'UN_2000_DS', 'UN_2005_DS', 'UN_2010_DS', 'UN_2015_DS',
       'UN_2020_DS', 'B_2010_E', 'F_2010_E', 'M_2010_E', 'A00_04B', 'A05_09B',
       'A10_14B', 'A15_19B', 'A20_24B', 'A25_29B', 'A30_34B', 'A35_39B',
       'A40_44B', 'A45_49B', 'A50_54B', 'A55_59B', 'A60_64B', 'A65PLUSB',
       'A65

We will be adding a `pop_density` variable to the fa24 data by joining observations with the observation in the population data which minimizes the Euclidian distance between `(Lat, Long)` and `(CENTROID_X, CENTROID_Y)`.


In [40]:
tree = cKDTree(
    pop_data[["CENTROID_X", "CENTROID_Y"]].values
)  # Using a CKD Tree to improve efficiency
distances, indices = tree.query(cur_data[["Long", "Lat"]].values)
pop_nearest = pop_data.iloc[indices][
    "UN_2020_DS"  # United Nations 2020 Adjusted Population Density Estimate
].reset_index(drop=True)
cur_data_reset = cur_data.reset_index(drop=True)

joined = pd.concat([cur_data_reset, pop_nearest], axis=1)
joined.head()

,Unnamed: 0,Date_of_observation,Species_name,Lat,Long,State_name,Leaves_fresh,Leaves_mature,Leaves_old,Flowers_bud,Flowers_open,Flowers_male,Flowers_Female,Fruits_unripe,Fruits_ripe,Fruits_open,Year,Week,Species_id,UN_2020_DS
0,1,2020-01-01,Indian Almond-Terminalia catappa,12.15386,75.22397,Kerala,2.0,0.0,0.0,-2.0,-2.0,-2.0,-2.0,-2.0,-2.0,-2.0,2020.0,0,1085.0,643.461262
1,2,2020-01-01,Indian Almond-Terminalia catappa,12.15386,75.22397,Kerala,2.0,0.0,0.0,-2.0,-2.0,-2.0,-2.0,-2.0,-2.0,-2.0,2020.0,0,1085.0,643.461262
2,3,2020-01-01,Fish-tail Palm-Caryota urens,12.14060,75.22145,Kerala,0.0,2.0,0.0,-2.0,-2.0,-2.0,-2.0,-2.0,-2.0,-2.0,2020.0,0,1019.0,643.461262
3,4,2020-01-01,Mast Tree-Monoon longifolium,12.14060,75.22145,Kerala,1.0,2.0,0.0,-2.0,-2.0,-2.0,-2.0,-2.0,-2.0,-2.0,2020.0,0,1065.0,643.461262
4,5,2020-01-01,Indian Almond-Terminalia catappa,12.14060,75.22145,Kerala,0.0,1.0,2.0,-2.0,-2.0,-2.0,-2.0,-2.0,-2.0,-2.0,2020.0,0,1085.0,643.461262


The Indian Ministry of Housing and Urban Affairs defines an urban area as an area that satisfies the following requirements:

(i) a minimum population of 5,000

(ii) at least 75% of male working population engaged in non-agricultural pursuits; and

(iii) a density of population of at least 400 persons per square kilometer.

[source](https://mohua.gov.in/pdf/5c80e2225a124Handbook%20of%20Urban%20Statistics%202019.pdf#page=26)

As we only have access to population density data, we will be relaxing this definition and focusing on the third (iii) requirement.


In [41]:
joined["is_urban"] = (joined["UN_2020_DS"] >= 400).astype(int)

In [42]:
joined.to_csv(
    "../../data/spring_2025_data/cleaned_alldata_v2_with_pop.csv", index=False
)